# Adversarial Patching via L0 Sparsity Gates

This notebook demonstrates a new interpretability technique: **Adversarial Patching**. 

Unlike standard feature attribution methods like Integrated Gradients which scale inputs from a baseline to compute importance, Adversarial Patching sets up an **optimization problem**. We aim to find the **minimal subset of tokens** that, when passed through the model, still produces the exact same prediction as the original full text.

All other tokens are "patched" (replaced) with a baseline token (e.g., `<unk>`). We achieve this by optimizing **L0 Sparsity Gates** on the token embeddings while minimizing the KL Divergence between the original and patched model outputs.

In [ ]:
import os
import sys
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.interpretability.feature_importance.adversarial_patching import AdversarialPatchingExplainer
from src.interpretability.viz.utils import plot_adversarial_patching, plot_text_heatmap

## 1. Load the Fine-Tuned Med-Gemma Classifier

In [ ]:
model_name = "google/medgemma-1.5-4b-it"
adapter_path = "../checkpoints/classifier_run/final_model"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading Base Model (bfloat16)...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=20,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Loading LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("Model ready!")

## 2. Load the Dataset

Let's load the real validation dataset (`tcga_reports_valid.csv`) to test our method on actual pathology reports.

In [ ]:
df_val = pd.read_csv('../tcga_reports_valid.csv')
print(f"Loaded {len(df_val)} validation reports.")

# Select a specific row to interpret (change index to explore different reports)
SAMPLE_INDEX = 42 
text_sample = df_val.iloc[SAMPLE_INDEX]['text']
true_label = df_val.iloc[SAMPLE_INDEX]['cancer_type']

print(f"\nSelected Report (True Label: {true_label}):\n")
print(text_sample[:1000] + ("..." if len(text_sample) > 1000 else ""))

## 3. Configure Optimization Parameters

Adversarial Patching operates by trading off between keeping the original prediction (`KL Divergence < kl_threshold`) and removing as many tokens as possible (`Sparsity`).

*   **`BASELINE_TYPE`**: Replaces dropped tokens with this token's embedding. Options: `"unk"`, `"pad"`, `"zero"`.
*   **`KL_THRESHOLD`**: The maximum acceptable difference between the original output distribution and the patched output distribution. A smaller number (e.g., `0.05`) forces the model to keep more tokens to stay accurate. A larger number allowed (e.g., `0.2`) drops more tokens.
*   **`SPARSITY_WEIGHT`**: Knob to explicitly push for more aggressive sparsification. Increase this (`2.0`, `5.0`) to force the gates closed more quickly if the KL divergence constraint is being met too easily.
*   **`USE_TASK_LOSS`**: If true, explicitly adds a Cross-Entropy loss targeted precisely at the original predicted class, serving as a powerful anchor to maintain the prediction, rather than just relying evenly on KL Divergence across all logits.
*   **`MAX_EPOCHS`**: Number of gradient steps to take for the L0 gates per sample.
*   **`LEARNING_RATE`**: Learning rate for the L0 gates.

In [ ]:
BASELINE_TYPE = "unk"  # Options: "unk", "pad", "zero"
KL_THRESHOLD = 0.05
SPARSITY_WEIGHT = 1.0  # Sparsity budget knob (increase to drop more tokens aggressively)
USE_TASK_LOSS = True   # Anchor predictions more aggressively using cross-entropy to the target class
MAX_EPOCHS = 200
LEARNING_RATE = 0.1

## 4. Run Optimization

Initialize the explainer and run the interpret loop.

In [ ]:
explainer = AdversarialPatchingExplainer(model, tokenizer)

results = explainer.interpret(
    input_text=text_sample,
    kl_threshold=KL_THRESHOLD,
    max_epochs=MAX_EPOCHS,
    lr=LEARNING_RATE,
    baseline_type=BASELINE_TYPE,
    sparsity_weight=SPARSITY_WEIGHT,
    use_task_loss=USE_TASK_LOSS
)

print(f"\nTarget Class Index (Predicted by model): {results['target_class_idx']}")
print(f"Patched Class Index (Predicted by patched model): {results['patched_class_idx']}")
print(f"KL Divergence + Task Loss Achieved: {results['kl_divergence']:.4f}")
print(f"Sparsity Rate Achieved: {results['sparsity_rate']*100:.2f}% tokens masked/patched.")

## 5. Visualize Results

We can visualize which tokens the optimizer decided were strictly necessary to keep (in green), and which it aggressively patched out/removed (in gray).

In [ ]:
plot_adversarial_patching(
    tokens=results["tokens"],
    scores=results["scores"],
    title=f"Adversarial Patching (KL Threshold: {KL_THRESHOLD}, Baseline: {BASELINE_TYPE})"
)

## 6. Continuous Gate Values (Heatmap)

While the final mask is a hard binary decision (keep vs. drop), the underlying optimization learns continuous "gate" values (alphas) between 0 and 1 before the threshold is applied. We can visualize these raw continuous probability values as a standard heatmap.

In [ ]:
plot_text_heatmap(
    tokens=results["tokens"],
    scores=results["gate_values"],
    title="Continuous Gate Values (Alpha)"
)